# Deformable DETR Training on Colab

This notebook trains the repository's Deformable DETR implementation with:

- checkpoints stored on Google Drive
- automatic resume from the latest checkpoint
- TensorBoard metrics instead of Weights & Biases
- configurable Waymo segment count and Drive checkpoint folder

Set `CHECKPOINT_DIR` to an empty string to train from scratch without saving checkpoints.

In [1]:
from google.colab import auth, drive

auth.authenticate_user()
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/yharidy/object_detection.git /content/object_detection
%cd /content/object_detection
!pip install -q -r requirements.txt
!pip install -q -e .

Cloning into '/content/object_detection'...
remote: Enumerating objects: 486, done.
remote: Counting objects: 100% (486/486), done.
remote: Compressing objects: 100% (342/342), done.
remote: Total 486 (delta 262), reused 336 (delta 132), pack-reused 0 (from 0)
Receiving objects: 100% (486/486), 5.76 MiB | 4.60 MiB/s, done.
Resolving deltas: 100% (262/262), done.
/content/object_detection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 45.6 MB/s eta 0:00

## Dataset partitions

This notebook uses Waymo's official partitions:

- `training`: used to update model weights.
- `validation`: used after each epoch to monitor generalization and select the best checkpoint or hyperparameters.
- `testing`: held back for the final evaluation after model and hyperparameters are frozen.

Set `TRAIN_NUM_SEGMENTS`, `VAL_NUM_SEGMENTS`, and `TEST_NUM_SEGMENTS` in the configuration cell to limit loading while debugging. `-1` means all available segments.

`CHECKPOINT_DIR` should point to a folder inside mounted Google Drive. Set it to `''` to disable checkpoint loading and saving.

In [3]:
import torch
from pathlib import Path

DATA_ROOT = 'gs://waymo_open_dataset_v_2_0_1'
TRAIN_SPLIT = 'training'
VAL_SPLIT = 'validation'
TEST_SPLIT = 'testing'
TRAIN_NUM_SEGMENTS = -1
VAL_NUM_SEGMENTS = -1
TEST_NUM_SEGMENTS = -1
VERSION = 6
CHECKPOINT_DIR = f"/content/drive/MyDrive/deformable_detr/checkpoints/v{VERSION}"
RESUME = True

BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 8
TARGET_SIZE = (480, 320)  # (width, height): ~1.5 aspect ratio close to Waymo's native images
NUM_EPOCHS = 40
NUM_CLASSES = 5
NUM_QUERIES = 300
NUM_LEVELS = 4
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
NUM_HEADS = 8
HIDDEN_DIM = 256
LEARNING_RATE = 2e-4            # base LR for the transformer (encoder/decoder)
BACKBONE_LEARNING_RATE = 2e-5   # 0.1x base LR, for the pretrained backbone only
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 0.1
CLASS_LOSS_WEIGHT = 1.0
BOX_LOSS_WEIGHT = 5.0
GIOU_LOSS_WEIGHT = 2.0
CHECKPOINT_EVERY_STEPS = 100
PREDICTIONS_EVERY_STEPS = 50
PREDICTION_SCORE_THRESHOLD = 0.5
MAX_PREDICTION_IMAGES = 4
USE_AMP = True
VALIDATE_EVERY_EPOCHS = 5
VALIDATE_AFTER_EPOCH = 10

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = USE_AMP and DEVICE.type == 'cuda'

print('Device:', DEVICE)
print('Train split:', TRAIN_SPLIT, 'segments:', TRAIN_NUM_SEGMENTS)
print('Validation split:', VAL_SPLIT, 'segments:', VAL_NUM_SEGMENTS)
print('Test split:', TEST_SPLIT, 'segments:', TEST_NUM_SEGMENTS)
print('Checkpoint directory:', CHECKPOINT_DIR or 'disabled')
print('TensorBoard directory:', str(Path(CHECKPOINT_DIR) / 'tensorboard') if CHECKPOINT_DIR else '/content/tensorboard/deformable_detr')
print('Resume enabled:', RESUME)
print('Automatic mixed precision:', USE_AMP)
print('Prediction logging interval:', PREDICTIONS_EVERY_STEPS)
print('Learning rate (transformer / backbone):', LEARNING_RATE, '/', BACKBONE_LEARNING_RATE)
print('Gradient clip norm:', GRAD_CLIP_NORM)


Device: cuda
Train split: training segments: -1
Validation split: validation segments: -1
Test split: testing segments: -1
Checkpoint directory: /content/drive/MyDrive/deformable_detr/checkpoints/v6
TensorBoard directory: /content/drive/MyDrive/deformable_detr/checkpoints/v6/tensorboard
Resume enabled: True
Automatic mixed precision: True
Prediction logging interval: 50
Learning rate (transformer / backbone): 0.0002 / 2e-05
Gradient clip norm: 0.1


In [4]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from src.datasets.camera_2d.collate import camera_2d_collate_fn
from src.datasets.camera_2d.dataset import Camera2DDataset
from src.datasets.camera_2d.transforms import Camera2DTransform
from src.domain.enums import CameraPosition
from src.models.detr.deformable_detr import DeformableDetr
from src.models.detr.detr_loss import DETRLoss
from src.sources.waymo.factory import build_waymo_loaders
from src.utils.metrics import evaluate_detr_model
from src.utils.box_utils import cxcywh_to_xyxy

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

cameras = [CameraPosition.FRONT]
transform = Camera2DTransform(target_image_size=TARGET_SIZE)

train_loaders = build_waymo_loaders(
    data_root=DATA_ROOT,
    split=TRAIN_SPLIT,
    cameras=cameras,
    load_camera_labels=True,
    num_segments=TRAIN_NUM_SEGMENTS,
)
val_loaders = build_waymo_loaders(
    data_root=DATA_ROOT,
    split=VAL_SPLIT,
    cameras=cameras,
    load_camera_labels=True,
    num_segments=VAL_NUM_SEGMENTS,
)

train_dataset = Camera2DDataset(
    loaders=train_loaders,
    cameras=cameras,
    transform=transform,
)
val_dataset = Camera2DDataset(
    loaders=val_loaders,
    cameras=cameras,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=camera_2d_collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=camera_2d_collate_fn,
)

print('Training segments:', len(train_loaders))
print('Training samples:', len(train_dataset))
print('Training batches per epoch:', len(train_loader))
print('Validation segments:', len(val_loaders))
print('Validation samples:', len(val_dataset))
print('Validation batches:', len(val_loader))
print('Test split is reserved for final evaluation after model selection:', TEST_SPLIT)

Training segments: 798
Training samples: 158081
Training batches per epoch: 79041
Validation segments: 202
Validation samples: 39987
Validation batches: 19994
Test split is reserved for final evaluation after model selection: testing


In [5]:
model = DeformableDetr(
    num_levels=NUM_LEVELS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    num_classes=NUM_CLASSES,
    num_queries=NUM_QUERIES,
    num_heads=NUM_HEADS,
    hidden_dim=HIDDEN_DIM,
    pretrained_backbone=True,
).to(DEVICE)

criterion = DETRLoss(
    image_width=TARGET_SIZE[0],
    image_height=TARGET_SIZE[1],
    num_classes=NUM_CLASSES,
    class_loss_weight=CLASS_LOSS_WEIGHT,
    box_loss_weight=BOX_LOSS_WEIGHT,
    giou_loss_weight=GIOU_LOSS_WEIGHT,
)

backbone_params = [p for n, p in model.named_parameters() if n.startswith('backbone.') and p.requires_grad]
transformer_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.') and p.requires_grad]

optimizer = torch.optim.AdamW(
    [
        {'params': backbone_params, 'lr': BACKBONE_LEARNING_RATE},
        {'params': transformer_params, 'lr': LEARNING_RATE},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[2, 5, 10], gamma=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

checkpoint_path = Path(CHECKPOINT_DIR) / 'latest.pt' if CHECKPOINT_DIR else None
log_dir = Path(CHECKPOINT_DIR) / 'tensorboard' if CHECKPOINT_DIR else Path('/content/tensorboard/deformable_detr')
if CHECKPOINT_DIR:
    Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(log_dir))

# start_epoch / start_batch describe where to resume
start_epoch = 0
start_batch = 0
start_val_batch = 0
val_running_loss = 0.0
global_step = 0

if RESUME and checkpoint_path is not None and checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    if checkpoint.get('scaler_state') is not None:
        scaler.load_state_dict(checkpoint['scaler_state'])
    if checkpoint.get('scheduler_state') is not None:
      scheduler.load_state_dict(checkpoint['scheduler_state'])
    start_epoch = checkpoint.get('next_epoch', 0)
    start_batch = checkpoint.get('next_batch', 0)
    start_val_batch = checkpoint.get('next_val_batch', 0)
    val_running_loss = checkpoint.get('val_running_loss', 0.0)
    global_step = checkpoint.get('global_step', 0)
    print(f'Resumed from epoch {start_epoch}, train batch {start_batch}, val batch {start_val_batch}, step {global_step}')
else:
    print('Training from scratch')

print(f'Backbone parameters: {sum(p.numel() for p in backbone_params):,} | Transformer parameters: {sum(p.numel() for p in transformer_params):,}')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 238MB/s]


Resumed from epoch 1, train batch 11877, val batch 0, step 91000
Backbone parameters: 11,936,296 | Transformer parameters: 13,901,344


In [6]:
def save_checkpoint(next_epoch, next_batch, global_step, next_val_batch=0, val_running_loss=0.0, keep_permanent=False):
    if checkpoint_path is None:
        return
    state = {
        'next_epoch': next_epoch,
        'next_batch': next_batch,
        'next_val_batch': next_val_batch,
        'val_running_loss': val_running_loss,
        'global_step': global_step,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict() if USE_AMP else None,
        'scheduler_state': scheduler.state_dict(),
        'config': {
            'num_classes': NUM_CLASSES,
            'num_queries': NUM_QUERIES,
            'target_size': TARGET_SIZE,
            'hidden_dim': HIDDEN_DIM,
            'checkpoint_dir': CHECKPOINT_DIR,
        },
    }
    temporary_path = checkpoint_path.with_suffix('.tmp')
    torch.save(state, temporary_path)
    temporary_path.replace(checkpoint_path)
    message = f'Saved checkpoint: {checkpoint_path}'
    if keep_permanent:
        epoch_path = Path(CHECKPOINT_DIR) / f'epoch_{next_epoch:04d}_step_{global_step:08d}.pt'
        torch.save(state, epoch_path)
        message += f' (+ permanent snapshot {epoch_path.name})'
    print(message)


def move_targets_to_device(batch):
    target_boxes = [boxes.to(DEVICE) for boxes in batch['boxes']]
    target_labels = [labels.to(DEVICE) for labels in batch['labels']]
    return target_boxes, target_labels


def evaluate_loss(model, data_loader, epoch, global_step, start_batch=0, running_loss=0.0):
    """Compute mean DETR loss over a labeled evaluation split."""
    was_training = model.training
    model.eval()
    total_loss = running_loss
    batch_count = start_batch
    with torch.no_grad():
        for batch_index, batch in enumerate(data_loader, start=start_batch):
            images = batch['image'].to(DEVICE, non_blocking=True)
            target_boxes, target_labels = move_targets_to_device(batch)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                decoder_output, _ = model(images, masks=None)
                _, intermediate_boxes, intermediate_logits = decoder_output
                loss = 0.0
                for boxes, logits in zip(intermediate_boxes, intermediate_logits):
                    layer_loss,_ = criterion(logits, boxes, target_labels, target_boxes)
                    loss += layer_loss

            loss_value = loss.float().item()
            total_loss += loss_value
            batch_count += 1

            if batch_index % 10 == 0:
                print(f'Eval Epoch {epoch + 1} | batch {batch_index + 1}/{batch_count + len(data_loader) - (batch_index + 1 - start_batch)} | loss {loss_value:.4f}')

            if CHECKPOINT_DIR and (batch_index + 1) % CHECKPOINT_EVERY_STEPS == 0:
                save_checkpoint(epoch, 0, global_step, next_val_batch=batch_index + 1, val_running_loss=total_loss)

    if was_training:
        model.train()
    return total_loss / max(1, batch_count)


def log_predictions_to_tensorboard(model, batch, writer, global_step):
    """Log denormalized images with ground-truth and predicted boxes."""
    was_training = model.training
    model.eval()
    images = batch['image'].to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            decoder_output, _ = model(images, masks=None)
            _, intermediate_boxes, intermediate_logits = decoder_output
            pred_boxes = intermediate_boxes[-1]
            pred_logits = intermediate_logits[-1]
        probabilities = pred_logits.float().softmax(dim=-1)
        pred_scores, pred_labels = probabilities[..., :-1].max(dim=-1)

    mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(3, 1, 1)
    scale = torch.tensor(
        [TARGET_SIZE[0], TARGET_SIZE[1], TARGET_SIZE[0], TARGET_SIZE[1]],
        device=DEVICE,
        dtype=pred_boxes.dtype,
    )

    for image_index in range(min(images.shape[0], MAX_PREDICTION_IMAGES)):
        image = (images[image_index] * std + mean).clamp(0, 1).cpu().permute(1, 2, 0).numpy()
        figure, axis = plt.subplots(figsize=(8, 8))
        axis.imshow(image)
        axis.set_axis_off()

        gt_boxes = cxcywh_to_xyxy(batch['boxes'][image_index]).cpu().numpy()
        gt_labels = batch['labels'][image_index].cpu().numpy()
        for box, label in zip(gt_boxes, gt_labels):
            x_min, y_min, x_max, y_max = box
            axis.add_patch(Rectangle(
                (x_min, y_min), x_max - x_min, y_max - y_min,
                fill=False, edgecolor='lime', linewidth=1.5,
            ))
            axis.text(x_min, y_min, f'GT {int(label)}', color='lime', fontsize=8,
                      bbox={'facecolor': 'black', 'alpha': 0.6, 'pad': 1})

        predicted_boxes = cxcywh_to_xyxy(pred_boxes[image_index]) * scale
        keep = pred_scores[image_index] >= PREDICTION_SCORE_THRESHOLD
        for box, label, score in zip(
            predicted_boxes[keep].cpu().numpy(),
            pred_labels[image_index][keep].cpu().numpy(),
            pred_scores[image_index][keep].cpu().numpy(),
        ):
            x_min, y_min, x_max, y_max = box
            axis.add_patch(Rectangle(
                (x_min, y_min), x_max - x_min, y_max - y_min,
                fill=False, edgecolor='red', linewidth=1.5,
            ))
            axis.text(x_min, y_max, f'Pred {int(label)} {score:.2f}', color='red', fontsize=8,
                      bbox={'facecolor': 'white', 'alpha': 0.7, 'pad': 1})

        axis.set_xlim(0, TARGET_SIZE[0])
        axis.set_ylim(TARGET_SIZE[1], 0)
        axis.set_title(f'Image {image_index}: green=GT, red=prediction')
        writer.add_figure('predictions', figure, global_step)
        plt.close(figure)

    if was_training:
        model.train()
    del images, pred_boxes, pred_logits, probabilities
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

In [7]:
model.train()
for epoch in range(start_epoch, NUM_EPOCHS):
    resume_this_epoch = (epoch == start_epoch and start_batch > 0)
    resume_val_this_epoch = (epoch == start_epoch and start_val_batch > 0)
    first_batch_index = start_batch if resume_this_epoch else 0

    epoch_loss = 0.0
    epoch_batch_count = 0

    if not resume_val_this_epoch:
        epoch_generator = torch.Generator().manual_seed(SEED + epoch)
        epoch_indices = torch.randperm(len(train_dataset), generator=epoch_generator).tolist()
        if resume_this_epoch:
            epoch_indices = epoch_indices[first_batch_index * BATCH_SIZE:]

        epoch_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=epoch_indices,
            num_workers=NUM_WORKERS,
            collate_fn=camera_2d_collate_fn,
        )

        for batch_index, batch in enumerate(epoch_loader, start=first_batch_index):
            images = batch['image'].to(DEVICE, non_blocking=True)
            target_boxes, target_labels = move_targets_to_device(batch)

            is_accum_start = (batch_index - first_batch_index) % GRAD_ACCUM_STEPS == 0
            if is_accum_start:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                decoder_output, _ = model(images, masks=None)
                _, intermediate_boxes, intermediate_logits = decoder_output
                loss = 0.0
                for boxes, logits in zip(intermediate_boxes, intermediate_logits):
                    layer_loss, loss_components = criterion(logits, boxes, target_labels, target_boxes)
                    loss += layer_loss

            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()

            is_accum_end = (
                (batch_index - first_batch_index + 1) % GRAD_ACCUM_STEPS == 0
                or batch_index + 1 == len(train_loader)
            )
            if is_accum_end:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()

            loss_value = loss.detach().item()
            epoch_loss += loss_value
            epoch_batch_count += 1
            global_step += 1
            writer.add_scalar('train/loss', loss_value, global_step)
            writer.add_scalar('train/loss_class', loss_components['class'], global_step)
            writer.add_scalar('train/loss_box', loss_components['box'], global_step)
            writer.add_scalar('train/loss_giou', loss_components['giou'], global_step)
            writer.add_scalar('train/learning_rate', optimizer.param_groups[-1]['lr'], global_step)

            if global_step % 10 == 0:
                print(f'Epoch {epoch + 1}/{NUM_EPOCHS} | batch {batch_index + 1}/{len(train_loader)} | loss {loss_value:.4f}')
                writer.flush()

            next_epoch = epoch
            next_batch = batch_index + 1
            if next_batch >= len(train_loader):
                next_epoch = epoch + 1
                next_batch = 0
            if CHECKPOINT_DIR and global_step % CHECKPOINT_EVERY_STEPS == 0:
                save_checkpoint(next_epoch, next_batch, global_step)
                writer.flush()

            if global_step % PREDICTIONS_EVERY_STEPS == 0:
                log_predictions_to_tensorboard(model, batch, writer, global_step)
                writer.flush()

            del images, target_boxes, target_labels, loss
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()

        mean_train_loss = epoch_loss / max(1, epoch_batch_count)
        writer.add_scalar('train/epoch_loss', mean_train_loss, epoch + 1)
        writer.flush()
    else:
        print(f'Resuming validation for epoch {epoch + 1} from batch {start_val_batch}...')

    v_start = start_val_batch if resume_val_this_epoch else 0
    v_loss = val_running_loss if resume_val_this_epoch else 0.0

    should_validate = resume_val_this_epoch or ((epoch + 1) >= VALIDATE_AFTER_EPOCH and (epoch + 1) % VALIDATE_EVERY_EPOCHS == 0)

    if should_validate:
        if v_start > 0:
            val_indices = list(range(len(val_dataset)))[v_start * BATCH_SIZE:]
            epoch_val_loader = DataLoader(
                val_dataset,
                batch_size=BATCH_SIZE,
                sampler=val_indices,
                num_workers=NUM_WORKERS,
                collate_fn=camera_2d_collate_fn,
            )
        else:
            epoch_val_loader = val_loader

        mean_val_loss = evaluate_loss(model, epoch_val_loader, epoch, global_step, start_batch=v_start, running_loss=v_loss)
        writer.add_scalar('validation/loss', mean_val_loss, epoch + 1)
        writer.flush()
        print(f'Finished epoch {epoch + 1}; validation loss: {mean_val_loss:.4f}')
    else:
        print(f'Finished epoch {epoch + 1}; skipping validation.')

    scheduler.step()
    save_checkpoint(epoch + 1, 0, global_step, keep_permanent=True)
    writer.flush()

print('Training finished. Run the next cell for final validation KPIs.')

Epoch 2/40 | batch 11887/79041 | loss 18.2250
Epoch 2/40 | batch 11897/79041 | loss 17.0223
Epoch 2/40 | batch 11907/79041 | loss 18.6742
Epoch 2/40 | batch 11917/79041 | loss 16.8378
Epoch 2/40 | batch 11927/79041 | loss 17.8414
Epoch 2/40 | batch 11937/79041 | loss 18.0447
Epoch 2/40 | batch 11947/79041 | loss 16.9803
Epoch 2/40 | batch 11957/79041 | loss 16.5934
Epoch 2/40 | batch 11967/79041 | loss 17.6165
Epoch 2/40 | batch 11977/79041 | loss 19.1414
Saved checkpoint: /content/drive/MyDrive/deformable_detr/checkpoints/v6/latest.pt
Epoch 2/40 | batch 11987/79041 | loss 15.5967
Epoch 2/40 | batch 11997/79041 | loss 18.2669
Epoch 2/40 | batch 12007/79041 | loss 18.4323
Epoch 2/40 | batch 12017/79041 | loss 18.6452
Epoch 2/40 | batch 12027/79041 | loss 16.1568
Epoch 2/40 | batch 12037/79041 | loss 17.0467
Epoch 2/40 | batch 12047/79041 | loss 17.8074
Epoch 2/40 | batch 12057/79041 | loss 15.7458
Epoch 2/40 | batch 12067/79041 | loss 10.3340
Epoch 2/40 | batch 12077/79041 | loss 16.221

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78602eba3920>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1671, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.13/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/usr/lib/python3.13/multiprocessing/popen_fork.py", line 41, in wait
    if not wait([self.sentinel], timeout):
  File "/usr/lib/python3.13/multiprocessing/connection.py", line 1174, in wait
    ready = selector.select(timeout)
  File "/usr/lib/python3.13/selectors.py", line 398, in select
    fd_event_list = self._selector.poll(timeout)
KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
print('Evaluating final model on the validation split...')
validation_metrics = evaluate_detr_model(
    model=model,
    data_loader=val_loader,
    device=DEVICE,
    image_width=TARGET_SIZE[0],
    image_height=TARGET_SIZE[1],
    num_classes=NUM_CLASSES,
    score_threshold=PREDICTION_SCORE_THRESHOLD,
    iou_threshold=0.5,
    use_amp=USE_AMP,
)

for metric_name, metric_value in validation_metrics.items():
    print(f'{metric_name}: {metric_value:.4f}')
    writer.add_scalar(f'validation/{metric_name}', metric_value, global_step)
writer.flush()
writer.close()